In [ ]:
from datasets import load_dataset, concatenate_datasets, load_from_disk
import json
from tree_sitter import Language, Parser
import tree_sitter_python

PYTHON_LANGUAGE = Language(tree_sitter_python.language())
parser = Parser()
parser.language = PYTHON_LANGUAGE

def norm_node_iterative_for_parser(code: str, parser: Parser):
    src = code.encode('utf8')
    tree = parser.parse(src)
    root_node = tree.root_node

    placeholder_state = {}
    stack = [root_node]
    results = {}

    while stack:
        node = stack[-1]
        t = node.type

        if t == 'identifier':
            name = src[node.start_byte:node.end_byte].decode('utf8')
            if name not in placeholder_state:
                placeholder_state[name] = f'VAR_{len(placeholder_state)}'
            results[node.id] = ('identifier', placeholder_state[name])
            stack.pop()
            continue

        if node.child_count == 0:
            text = src[node.start_byte:node.end_byte].decode('utf8')
            results[node.id] = (t, text)
            stack.pop()
            continue

        all_children_processed = True
        children_to_visit = []
        for child in node.children:
            if child.id not in results:
                all_children_processed = False
                children_to_visit.append(child)

        if all_children_processed:
            normalized_children = tuple(results[child.id] for child in node.children)
            results[node.id] = (t, normalized_children)
            stack.pop()
        else:
            for child in reversed(children_to_visit):
                stack.append(child)

    return results[root_node.id]

def dedupe_sols_ast(sols):
    seen = set()
    unique = []
    for code in sols:
        try:
            norm = norm_node_iterative_for_parser(code, parser)
        except Exception:
            norm = ('parse_error', ' '.join(str(code).split()))
        if norm not in seen:
            seen.add(norm)
            unique.append(code)
    return unique


In [ ]:
code_contests = load_dataset("deepmind/code_contests")
code_contests_merged = concatenate_datasets([
  code_contests['train'],
  code_contests['test'],
  code_contests['valid']
])

code_contests_merged_filt_cols = code_contests_merged.remove_columns([
    'description', 'incorrect_solutions', 'is_description_translated', 'untranslated_description'
])

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
LANGUAGE_CODES = {'UNKNOWN':0, 'PYTHON':1, 'CPP':2, 'PYTHON3':3, 'JAVA':4}
def simple_norm(code):
    if not isinstance(code, str):
        return str(code)
    return ' '.join(code.split())
def dedupe_sols(sols):
    seen = set()
    unique = []
    for s in sols:
        n = simple_norm(s)
        if n not in seen:
            seen.add(n)
            unique.append(s)
    return unique
def filt_sols_for_lang_factory(lang_code):
    def filt(row):
        new_sols = {'language': [], 'solution': []}
        all_langs = row['solutions']['language']
        all_sols = row['solutions']['solution']
        seen = set()
        for i in range(len(all_langs)):
            lang = all_langs[i]
            sol = all_sols[i]
            n = simple_norm(sol)
            if lang == lang_code and n not in seen:
                seen.add(n)
                new_sols['language'].append(lang)
                new_sols['solution'].append(sol)
        return {'solutions': new_sols}
    return filt

In [ ]:
lang_code = LANGUAGE_CODES['PYTHON3']
filt = filt_sols_for_lang_factory(lang_code)
py_set = code_contests_merged_filt_cols.map(filt, batched=False)
py_set = py_set.filter(lambda x: len(x['solutions']['language']) > 0, batched=False)
cols_to_remove = [c for c in ['input_file','output_file'] if c in py_set.column_names]
if cols_to_remove:
    py_set = py_set.remove_columns(cols_to_remove)

def keep_sols_only(row):
    if isinstance(row['solutions'], dict) and 'solution' in row['solutions']:
        return {'solutions': row['solutions']['solution']}
    return {'solutions': row['solutions']}

py_set = py_set.map(keep_sols_only, batched=False)
py_set = py_set.map(lambda x: {'solutions': dedupe_sols_ast(x['solutions'])}, batched=False)
py_set.to_json('code_contests_python3.jsonl')
print('Python3 dataset (AST dedupe) created')
print('Rows:', len(py_set))
print('Total solutions:', sum(len(r['solutions']) for r in py_set))

Map:   0%|          | 0/13610 [00:00<?, ? examples/s]

Filter:   0%|          | 0/13610 [00:00<?, ? examples/s]

Map:   0%|          | 0/8356 [00:00<?, ? examples/s]

Parameter 'function'=<function <lambda> at 0x131e81800> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/8356 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Python3 dataset (AST dedupe) created
Rows: 8356
Total solutions: 1390531
Total solutions: 1390531
